# <center> Split Data </center>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

C:\Users\ELITEBOOK\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv(r"..\data\interim\annotated_data.csv")

In [3]:
def replace_column_with_less_than_1_occurrence(df:pd.DataFrame, column:str) -> pd.DataFrame:
    # df = df.copy() --> je ne veux pas exploser la mémoire.
    value_counts = df[column].value_counts()
    value_with_less_than_1_occurence = [value for value in value_counts.index if value_counts[value]<=1]
    __is_less_than_one = df[column].isin(value_with_less_than_1_occurence)
    df[column] = df[column].mask(__is_less_than_one, value_counts.index[0])
    return df

In [4]:
# Stratify guarantees that the *proportions* of a category are preserved
# in both train and test. Stratifying only on "seniority" says nothing about
# "home_office_policy", so the minority "hybrid" class can end up under/over
# represented in test just by chance.
#
# sklearn's train_test_split only accepts ONE stratify array, so to respect
# both columns at once we build a single combined key (e.g. "confirme_on_site",
# "lead_hybrid", ...) and stratify on that instead.

df = replace_column_with_less_than_1_occurrence(df, "seniority")

# A handful of rows have no home_office_policy value at all (NaN). "+" would
# propagate that NaN into the whole key, so give missing values their own
# "unknown" bucket instead of silently dropping/breaking those rows.
df["strata"] = df["seniority"] + "_" + df["home_office_policy"].fillna("unknown")

# Some seniority x home_office_policy combos only occur once (e.g. senior_hybrid,
# junior_hybrid). stratify requires every class to have at least 2 members
# (one for train, one for test), so we reuse the same merge logic on the
# combined key before splitting.
df = replace_column_with_less_than_1_occurrence(df, "strata")

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["strata"],
    random_state=42
)

df_train = df_train.drop(columns=["strata"])
df_test = df_test.drop(columns=["strata"])

In [5]:
df_train.to_csv(r"..\data\interim\train.csv", index=False)
df_test.to_csv(r"..\data\interim\test.csv", index=False)

In [6]:
# Sanity check: home_office_policy should now be proportionally represented
# in test (previously only 2 hybrid rows made it into test).
print(df_train["home_office_policy"].value_counts(normalize=True))
print(df_test["home_office_policy"].value_counts())

home_office_policy
on_site    0.848101
hybrid     0.151899
Name: proportion, dtype: float64
home_office_policy
on_site    16
hybrid      4
Name: count, dtype: int64
